In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import arviz as az
import xarray as xr

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
INFILE = "output_files/inference_ASV_annotated.nc"
OUTDIR = Path("export_for_R")

BASELINE_DAY = "day_01"
HDI_PROB = 0.95

RHAT_THRESHOLD = 1.05
ESS_TAIL_THRESHOLD = 400


# -----------------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------------
def asv_id_from_label(label):
    """Extract ASV ID from: 'ASV_13 | g__Cloacibacterium'."""
    return str(label).split(" | ", 1)[0].strip()


def to_dataarray(x, name):
    """Convert ArviZ output safely to one DataArray."""
    if isinstance(x, xr.Dataset):
        if len(x.data_vars) != 1:
            raise ValueError(f"Expected one variable, found: {list(x.data_vars)}")
        x = next(iter(x.data_vars.values()))

    if not isinstance(x, xr.DataArray):
        raise TypeError(f"Expected xarray.DataArray, got {type(x)}")

    return x.rename(name)


def to_long(da, value_name):
    """Convert an xarray DataArray to a tidy data frame."""
    return da.to_dataframe(name=value_name).reset_index()


def day_sort_key(day):
    """Sort day_01, day_02, ..., day_22 numerically."""
    match = re.search(r"(\d+)", str(day))
    return int(match.group(1)) if match else 9999


def find_day_levels(covariates):
    """Extract levels from terms such as Day[T.day_15]."""
    days = []

    for covariate in covariates:
        match = re.match(r"Day\[T\.(.+?)\]$", str(covariate))
        if match:
            days.append(match.group(1))

    return days


def finite_values(x):
    """Return finite numerical values from an xarray object."""
    if isinstance(x, xr.Dataset):
        values = x.to_array().values.ravel()
    else:
        values = x.values.ravel()

    return values[np.isfinite(values)]


def compact_mcmc_qc(parameter_name, posterior_da):
    """One-row global convergence summary for a posterior parameter."""
    rhat = finite_values(az.rhat(posterior_da))
    ess_bulk = finite_values(az.ess(posterior_da, method="bulk"))
    ess_tail = finite_values(az.ess(posterior_da, method="tail"))

    return {
        "parameter": parameter_name,
        "n_elements": len(rhat),
        "max_r_hat": np.max(rhat) if len(rhat) else np.nan,
        "min_ess_bulk": np.min(ess_bulk) if len(ess_bulk) else np.nan,
        "min_ess_tail": np.min(ess_tail) if len(ess_tail) else np.nan,
        "n_rhat_ge_1.05": np.sum(rhat >= RHAT_THRESHOLD),
        "n_ess_tail_le_400": np.sum(ess_tail <= ESS_TAIL_THRESHOLD),
    }


# -----------------------------------------------------------------------------
# PREPARE OUTPUT DIRECTORY
# -----------------------------------------------------------------------------
if OUTDIR.exists() and any(OUTDIR.iterdir()):
    raise FileExistsError(
        f"'{OUTDIR}' already contains files. "
        "Delete or rename the old export_for_R folder before running this script."
    )

OUTDIR.mkdir(exist_ok=True)
DIAGDIR = OUTDIR / "diagnostics"
DIAGDIR.mkdir(exist_ok=True)


# -----------------------------------------------------------------------------
# LOAD POSTERIOR
# -----------------------------------------------------------------------------
inference = az.from_netcdf(INFILE)

if "beta_var" not in inference.posterior:
    raise RuntimeError(
        "beta_var is missing. Available posterior variables: "
        f"{list(inference.posterior.data_vars)}"
    )

beta = inference.posterior["beta_var"]

if not {"chain", "draw", "covariate", "feature"}.issubset(beta.dims):
    raise RuntimeError(
        "beta_var must contain chain, draw, covariate, and feature dimensions."
    )

covariates = [str(x) for x in beta.coords["covariate"].values]
features = [str(x) for x in beta.coords["feature"].values]
feature_to_asv = {feature: asv_id_from_label(feature) for feature in features}


# -----------------------------------------------------------------------------
# BUILD HC − NC CONTRASTS FOR EACH DAY
# -----------------------------------------------------------------------------
diet_term = "Diet[T.HC]"

if diet_term not in covariates:
    raise RuntimeError(f"Missing expected coefficient: {diet_term}")

model_days = find_day_levels(covariates)

if not model_days:
    raise RuntimeError("No Day[T.day_*] coefficients detected.")

days = [BASELINE_DAY] + sorted(
    [day for day in model_days if day != BASELINE_DAY],
    key=day_sort_key
)


def coefficient(term):
    """Extract one posterior coefficient: chain × draw × feature."""
    return beta.sel(covariate=term).reset_coords(drop=True)


delta_base = coefficient(diet_term)
contrasts = {BASELINE_DAY: delta_base}

for day in days:
    if day == BASELINE_DAY:
        continue

    interaction = f"Diet[T.HC]:Day[T.{day}]"

    if interaction not in covariates:
        raise RuntimeError(
            f"Missing interaction coefficient required for {day}: {interaction}"
        )

    contrasts[day] = delta_base + coefficient(interaction)

# Dimensions: chain × draw × feature × day
delta_all = xr.concat(
    [contrasts[day] for day in days],
    dim="day"
).assign_coords(day=days)


# -----------------------------------------------------------------------------
# EXPORT 1: MAIN POSTERIOR RESULTS TABLE
# -----------------------------------------------------------------------------
mean_da = delta_all.mean(dim=("chain", "draw")).rename("mean")

p_positive = (delta_all > 0).mean(dim=("chain", "draw"))
p_negative = (delta_all < 0).mean(dim=("chain", "draw"))
pd_da = xr.apply_ufunc(np.maximum, p_positive, p_negative).rename("PD")

hdi = to_dataarray(az.hdi(delta_all, hdi_prob=HDI_PROB), "hdi")

hdi_label = f"{HDI_PROB:.2f}"
hdi_lower_name = f"hdi_lower_{hdi_label}"
hdi_upper_name = f"hdi_upper_{hdi_label}"

hdi_lower = hdi.sel(hdi="lower").rename(hdi_lower_name)
hdi_upper = hdi.sel(hdi="higher").rename(hdi_upper_name)

results_long = (
    to_long(mean_da, "mean")
    .merge(to_long(pd_da, "PD"), on=["feature", "day"])
    .merge(to_long(hdi_lower, hdi_lower_name), on=["feature", "day"])
    .merge(to_long(hdi_upper, hdi_upper_name), on=["feature", "day"])
)

results_long["asv_id"] = results_long["feature"].map(feature_to_asv)

results_long = results_long[
    [
        "asv_id",
        "feature",
        "day",
        "mean",
        "PD",
        hdi_lower_name,
        hdi_upper_name,
    ]
].sort_values(["day", "asv_id"])

results_path = OUTDIR / "HCminusNC_contrasts_posterior_summary_LONG.tsv"

results_long.to_csv(
    results_path,
    sep="\t",
    index=False
)


# -----------------------------------------------------------------------------
# EXPORT 2: ASV × DAY CONTRAST DIAGNOSTICS
# -----------------------------------------------------------------------------
rhat_da = to_dataarray(az.rhat(delta_all), "r_hat")
ess_tail_da = to_dataarray(
    az.ess(delta_all, method="tail"),
    "ess_tail"
)
ess_bulk_da = to_dataarray(
    az.ess(delta_all, method="bulk"),
    "ess_bulk"
)

diagnostics_long = (
    to_long(rhat_da, "r_hat")
    .merge(to_long(ess_tail_da, "ess_tail"), on=["feature", "day"])
    .merge(to_long(ess_bulk_da, "ess_bulk"), on=["feature", "day"])
)

diagnostics_long["asv_id"] = diagnostics_long["feature"].map(feature_to_asv)

diagnostics_long = diagnostics_long[
    [
        "asv_id",
        "feature",
        "day",
        "r_hat",
        "ess_tail",
        "ess_bulk",
    ]
].sort_values(["day", "asv_id"])

diagnostics_path = DIAGDIR / "delta_contrast_diagnostics_LONG.tsv"

diagnostics_long.to_csv(
    diagnostics_path,
    sep="\t",
    index=False
)


# -----------------------------------------------------------------------------
# EXPORT 3: COMPACT GLOBAL MCMC CONVERGENCE SUMMARY
# -----------------------------------------------------------------------------
qc_rows = [compact_mcmc_qc("beta_var", beta)]

if "inv_disp" in inference.posterior:
    qc_rows.append(
        compact_mcmc_qc(
            "inv_disp",
            inference.posterior["inv_disp"]
        )
    )

qc_summary = pd.DataFrame(qc_rows)

qc_path = DIAGDIR / "MCMC_convergence_summary.tsv"

qc_summary.to_csv(
    qc_path,
    sep="\t",
    index=False
)


# -----------------------------------------------------------------------------
# REPORT
# -----------------------------------------------------------------------------
print("\nExport complete. Three files written:")
print(f"1. {results_path}")
print(f"2. {diagnostics_path}")
print(f"3. {qc_path}")

print("\nKey contrast QC:")
print(f"Maximum contrast R-hat: {diagnostics_long['r_hat'].max():.4f}")
print(f"Minimum contrast ESS-tail: {diagnostics_long['ess_tail'].min():.0f}")



Export complete. Three files written:
1. export_for_R/HCminusNC_contrasts_posterior_summary_LONG.tsv
2. export_for_R/diagnostics/delta_contrast_diagnostics_LONG.tsv
3. export_for_R/diagnostics/MCMC_convergence_summary.tsv

Key contrast QC:
Maximum contrast R-hat: 1.0122
Minimum contrast ESS-tail: 854
